In [14]:
import os
import fitz  # PyMuPDF
from pathlib import Path

input_root = 'pdf sources'
output_root = 'slavery_text'  # or whatever you want to call your output folder

for root, dirs, files in os.walk(input_root):
    for file in files:
        if file.lower().endswith('.pdf'):
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, input_root)
            path_parts = Path(rel_path).parts  # splits by folder

            # Infer document_type and year from path
            # Example: PolicyArchive\kamerstukken\2005\10\14\bijlage-radio-nederland-wereldomroep-meerjarenplan-2004-2008.pdf
            # So path_parts = ('kamerstukken', '2005', '10', '14', 'bijlage-radio-nederland-wereldomroep-meerjarenplan-2004-2008.pdf')
            try:
                document_type = path_parts[0]
                year = path_parts[1]
                filename = Path(file).stem
            except IndexError:
                print(f"Path {rel_path} too short to extract document_type and year. Skipping.")
                continue

            # Open PDF and export per page
            try:
                pdf = fitz.open(full_path)
            except Exception as e:
                print(f"Error opening {full_path}: {e}")
                continue

            for page_number in range(pdf.page_count):
                page = pdf.load_page(page_number)
                text = page.get_text()
                #out_dir = os.path.join(output_root, document_type, year, filename)
                out_dir = os.path.join(output_root, filename)
                out_path = os.path.join(out_dir, f"{filename}.{page_number+1}.txt")
         # Ensure the directory exists for every file
                os.makedirs(os.path.dirname(out_path), exist_ok=True)
                with open(out_path, "w", encoding="utf-8") as f:
                   f.write(text)
            pdf.close()
            print(f"Processed {full_path}")

print("Done.")


Path Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen.pdf too short to extract document_type and year. Skipping.
Path Amsterdam en het slavernijverleden.pdf too short to extract document_type and year. Skipping.
Path Jouwe e.a. - Slavernij en de stad Utrecht.pdf too short to extract document_type and year. Skipping.
Path ketenen van het verleden.pdf too short to extract document_type and year. Skipping.
Path Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf too short to extract document_type and year. Skipping.
Path ZWART_MANIFEST.pdf too short to extract document_type and year. Skipping.
Done.


In [1]:
import os
import fitz
from pathlib import Path
from tqdm import tqdm
from joblib import Parallel, delayed

input_root = "pdf sources"
output_root = "slavery_text"  # or whatever you want to call your output folder
error_log = "error_log.txt"
N_JOBS = 1  # Set to the number of CPU cores you want to use
FLAT_OUTPUT = True  # Set to False for nested folders


def already_extracted(out_dir, filename, page_count):
    for i in range(1, page_count + 1):
        if not os.path.exists(os.path.join(out_dir, f"{filename}.{i}.txt")):
            return False
    return True

def process_pdf(full_path):
    try:
        filename = Path(full_path).stem
        if FLAT_OUTPUT:
            out_dir = output_root  # No per-pdf subfolder
        else:
            # Try to mimic old logic, fallback to "unknown" if missing folders
            rel_path = os.path.relpath(full_path, input_root)
            path_parts = Path(rel_path).parts
            document_type = path_parts[0] if len(path_parts) > 1 else "unknown_type"
            year = path_parts[1] if len(path_parts) > 2 else "unknown_year"
            out_dir = os.path.join(output_root, document_type, year, filename)

        try:
            pdf = fitz.open(full_path)
        except Exception as e:
            print(f"FAILED TO OPEN {full_path}: {e}")
            return f"FAILED TO OPEN {full_path}: {e}\n"
        print(f"Processing: {full_path} | Pages: {pdf.page_count}")
        if FLAT_OUTPUT:
            # Check extraction in flat output mode
            def already_extracted_flat(root, fname, page_count):
                for i in range(1, page_count + 1):
                    if not os.path.exists(os.path.join(root, f"{fname}.{i}.txt")):
                        return False
                return True
            skip = already_extracted_flat(out_dir, filename, pdf.page_count)
        else:
            skip = already_extracted(out_dir, filename, pdf.page_count)
        if skip:
            print(f"Already extracted: {out_dir}")
            pdf.close()
            return None
        for page_number in range(pdf.page_count):
            page = pdf.load_page(page_number)
            text = page.get_text()
            if FLAT_OUTPUT:
                out_path = os.path.join(out_dir, f"{filename}.{page_number+1}.txt")
            else:
                out_path = os.path.join(out_dir, f"{filename}.{page_number+1}.txt")
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            if not os.path.exists(out_path) or os.path.getsize(out_path) == 0:
                with open(out_path, "w", encoding="utf-8") as f:
                    f.write(text)
        pdf.close()
        return None
    except Exception as e:
        print(f"WORKER CRASHED {full_path}: {e}")
        return f"WORKER CRASHED {full_path}: {e}\n"


if __name__ == "__main__":
    # Gather all PDFs first
    pdf_files = []
    for root, dirs, files in os.walk(input_root):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_files.append(os.path.join(root, file))

    print(f"Found {len(pdf_files)} PDF files.")

    results = Parallel(n_jobs=N_JOBS, backend='loky')(
        delayed(process_pdf)(fname) for fname in tqdm(pdf_files)
    )

    # Write errors
    errors = [r for r in results if r]
    if errors:
        with open(error_log, "w", encoding="utf-8") as elog:
            elog.writelines(errors)
    print("Extraction done. Errors (if any) in error_log.txt")


Found 9 PDF files.


 44%|████▍     | 4/9 [00:00<00:00, 35.79it/s]

Processing: pdf sources\Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen.pdf | Pages: 480
Already extracted: slavery_text
Processing: pdf sources\Amsterdam en het slavernijverleden.pdf | Pages: 108
Already extracted: slavery_text
Processing: pdf sources\Jouwe e.a. - Slavernij en de stad Utrecht.pdf | Pages: 328
Already extracted: slavery_text
Processing: pdf sources\ketenen van het verleden.pdf | Pages: 272
Already extracted: slavery_text
Processing: pdf sources\Nimako et al_2020_Een rapport met betrekking tot het onderzoeksproject over de periode voor de.pdf | Pages: 166
Already extracted: slavery_text
Processing: pdf sources\Racisme bij het Ministerie van Buitenlandse Zaken. Een verkennend onderzoek.pdf | Pages: 110


 89%|████████▉ | 8/9 [00:00<00:00, 11.91it/s]

Processing: pdf sources\toc_en_intro.pdf | Pages: 18
Processing: pdf sources\Voortgangsrapportage+-+Ervaringen+en+lessen+om+discriminatie+in+publieke+dienstverlening+te+voorkomen+en+te+bestrijden.pdf | Pages: 52


100%|██████████| 9/9 [00:00<00:00, 14.74it/s]

Processing: pdf sources\ZWART_MANIFEST.pdf | Pages: 48
Already extracted: slavery_text
Extraction done. Errors (if any) in error_log.txt


In [12]:
print(already_extracted("some_output_dir", "filename", 12))


False


In [10]:
import os
import fitz
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from tqdm import tqdm
import multiprocessing

input_root = "PolicyArchive"
output_root = "Policyarchive_text"
error_log = "error_log.txt"
MAX_WORKERS = 3  # Try low first

def already_extracted(out_dir, filename, page_count):
    for i in range(1, page_count + 1):
        if not os.path.exists(os.path.join(out_dir, f"{filename}.{i}.txt")):
            return False
    return True

def process_pdf(full_path):
    try:
        rel_path = os.path.relpath(full_path, input_root)
        path_parts = Path(rel_path).parts
        document_type = path_parts[0]
        year = path_parts[1]
        filename = Path(full_path).stem
        out_dir = os.path.join(output_root, document_type, year, filename)
        try:
            pdf = fitz.open(full_path)
        except Exception as e:
            return f"FAILED TO OPEN {full_path}: {e}\n"
        if already_extracted(out_dir, filename, pdf.page_count):
            pdf.close()
            return None
        for page_number in range(pdf.page_count):
            page = pdf.load_page(page_number)
            text = page.get_text()
            out_path = os.path.join(out_dir, f"{filename}.{page_number+1}.txt")
            os.makedirs(os.path.dirname(out_path), exist_ok=True)
            if not os.path.exists(out_path) or os.path.getsize(out_path) == 0:
                with open(out_path, "w", encoding="utf-8") as f:
                    f.write(text)
        pdf.close()
        return None
    except Exception as e:
        return f"WORKER CRASHED {full_path}: {e}\n"

def run_parallel(files):
    errors = []
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(tqdm(executor.map(process_pdf, files), total=len(files), desc="Processing PDFs (parallel)"))
        for result, fname in zip(results, files):
            if result:
                errors.append((fname, result))
                if len(errors) >= 50:
                    with open(error_log, "a", encoding="utf-8") as elog:
                        elog.writelines([e[1] for e in errors])
                    errors = []
    if errors:
        with open(error_log, "a", encoding="utf-8") as elog:
            elog.writelines([e[1] for e in errors])
    return [e[0] for e in errors]  # Return files that errored

if __name__ == "__main__":
    multiprocessing.set_start_method('spawn', force=True)  # Use safest start method on Windows

    pdf_files = []
    for root, dirs, files in os.walk(input_root):
        for file in files:
            if file.lower().endswith('.pdf'):
                pdf_files.append(os.path.join(root, file))

    print(f"Found {len(pdf_files)} PDF files.")

    # 1st pass, try parallel
    crashed_files = run_parallel(pdf_files)

    # If any files failed, try sequentially (one-by-one) for robustness
    if crashed_files:
        print(f"\nRetrying {len(crashed_files)} crashed files sequentially...")
        for fname in tqdm(crashed_files, desc="Retrying (sequential)"):
            res = process_pdf(fname)
            if res:
                with open(error_log, "a", encoding="utf-8") as elog:
                    elog.write(res)

    print("Extraction done. Errors (if any) in error_log.txt")


Found 5953 PDF files.


Processing PDFs (parallel):   0%|          | 0/5953 [00:00<?, ?it/s]


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [1]:
import os
import re

def extract_toc_titles(text):
    """
    Heuristically extracts possible TOC lines from the first 100 lines.
    Only picks lines that look like a chapter/article (not page numbers, not blank).
    """
    lines = text.splitlines()
    toc_lines = []
    in_toc = False
    for line in lines[:200]:
        line = line.strip()
        # Start TOC at line with 'Inhoudsopgave'
        if 'inhoud' in line.lower():
            in_toc = True
            continue
        if in_toc:
            # Stop if empty line or too short
            if not line or len(line) < 5:
                continue
            # Stop TOC if the line looks like a section break
            if re.match(r"^\s*Deel\s+\d+", line):
                continue
            # End TOC at first number (page)
            if re.match(r'^\d+$', line):
                break
            # Heuristic: line is TOC if mostly words, not numbers
            if sum(c.isalpha() for c in line) > 5:
                toc_lines.append(line)
            # If a page number at end, remove it
            if re.search(r'\d+$', line):
                toc_lines[-1] = re.sub(r'\s*\d+\s*$', '', toc_lines[-1])
        # End after 30 lines of TOC
        if in_toc and len(toc_lines) > 30:
            break
    # Clean up: remove doubles, empty, short, etc.
    toc_lines = [l.strip() for l in toc_lines if len(l.strip()) > 7]
    return toc_lines

def chapter_regex_from_titles(titles):
    """
    Build regex pattern that matches any title as a chapter boundary.
    """
    esc_titles = [re.escape(title) for title in titles]
    pattern = r'(' + '|'.join(esc_titles) + r')'
    return pattern

def split_text_on_titles(text, titles):
    """
    Split main text using TOC titles as boundaries.
    Returns list of (title, chunk) tuples.
    """
    # Build one big regex pattern
    pattern = chapter_regex_from_titles(titles)
    splits = re.split(pattern, text, flags=re.IGNORECASE)
    # Remove any preamble before first title
    if len(splits) > 1:
        splits = splits[1:]
    # Pair each chunk with the title that comes before it
    chapters = []
    for i in range(0, len(splits)-1, 2):
        title = splits[i].strip()
        chunk = splits[i+1].strip()
        chapters.append((title, chunk))
    return chapters

# Main: Process all files in folder
document_path = "sources"
all_docs_chapters = {}

for fname in sorted(os.listdir(document_path)):
    if fname.lower().endswith(".txt"):
        with open(os.path.join(document_path, fname), encoding="utf-8") as f:
            text = f.read()
        titles = extract_toc_titles(text)
        chapters = split_text_on_titles(text, titles)
        all_docs_chapters[fname] = chapters
        print(f"{fname}: {len(chapters)} chapters detected.")
        print([t for t, _ in chapters])

# Example: To get all chapter texts for a file:
# chapters = all_docs_chapters['staatenslavernij.txt']  # or any file name
# for title, chunk in chapters:
#     print(f"== {title} ==\n{chunk[:300]}...\n")


Allen e.a. - 2023 - Staat en slavernij het Nederlandse koloniale slavernijverleden en zijn doorwerkingen__segment_01.txt: 298 chapters detected.
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', ''

In [4]:
# Verzamel alle hoofdstukteksten uit alle documenten:
all_chapter_texts = []
for chapters in all_docs_chapters.values():
    for title, chunk in chapters:
        all_chapter_texts.append(chunk)
